# 02. Model Development and Evaluation
## Restaurant Order Demand Prediction

### Purpose
This notebook illustrates model architecture development, chronological train-validation-test splitting,
baseline heuristic fitting, and tree-based gradient boosting regressor training.

### Validation Strategy
To prevent temporal leakage, data is partitioned chronologically:
- Train: Weeks 1 to 125
- Validation: Weeks 126 to 135
- Test: Weeks 136 to 145

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from restaurant_demand.models.train import ModelTrainer
from restaurant_demand.models.predict import DemandPredictor
from restaurant_demand.config import load_data_config, load_model_config

### 1. Training Pipeline Execution
Invoking the production pipeline trainer from the package ensures consistent preprocessing and feature generation.

In [ ]:
trainer = ModelTrainer(base_dir=project_root)
metrics = trainer.train_pipeline()

print('Test Set Performance Summary:')
display(pd.DataFrame([
    {'Model': 'Group Median Baseline', **metrics['baseline']['test']},
    {'Model': 'HistGradientBoosting', **metrics['candidate_hist_gradient_boosting']['test']},
]))

### 2. Error Analysis and Residual Diagnostics

In [ ]:
predictor = DemandPredictor(base_dir=project_root)
test_clean_path = project_root / 'data' / 'processed' / 'test.csv'
test_df = pd.read_csv(test_clean_path)

scored_test = predictor.predict_dataframe(test_df)
scored_test['residual'] = scored_test['num_orders'] - scored_test['predicted_demand']

print(f'Mean Residual: {scored_test["residual"].mean():.2f}')
print(f'Median Residual: {scored_test["residual"].median():.2f}')

plt.figure(figsize=(9, 5))
sns.histplot(scored_test['residual'], bins=60, kde=True, color='purple')
plt.axvline(0, color='red', linestyle='--')
plt.title('Held-Out Test Set Residual Error Distribution')
plt.xlabel('Residual (Actual - Predicted Orders)')
plt.show()

### 3. Conclusion
- HistGradientBoosting accurately captures non-linear promotional spikes and seasonal cycles.
- The persisted pipeline is versioned and ready for real-time serving.